In [14]:
%cd NISP-Dataset/Hindi_master

[Errno 2] No such file or directory: 'NISP-Dataset/Hindi_master'
/data/himanshu/NISP-Dataset/Hindi_master


/home/himanshu/.local/lib/python3.10/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


In [3]:
pip install google-generativeai

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 KB 1.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 12.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.2/173.2 KB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 31.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 KB 1.0 MB/s eta 0:00:00ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 KB 7.9 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.7/300.7 KB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 31.2 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.31.1
    Uninstalling protobuf-6.31.1:
      Successfully uninstalled protobuf-6.31.1
Note: you may need to restart the kernel to use updated packa

In [15]:
from getpass import getpass
import os

os.environ["GEMINI_API_KEY"] = getpass("Enter API key: ")

Enter API key:  ········


In [21]:
import google.generativeai as genai
import os

genai.configure(api_key=os.environ["GEMINI_API_KEY"])

model = genai.GenerativeModel("gemini-2.0-flash")

In [17]:
ALLOWED_STATES = [
    "Uttar_Pradesh", "Rajasthan", "Madhya_Pradesh", "Delhi",
    "Maharashtra", "Bihar", "Uttarakhand", "Chhattisgarh",
    "Haryana", "Karnataka", "Jharkhand", "West_Bengal",
    "Gujarat", "Jammu_&_Kashmir", "Odisha", "Panjab",
    "Himachal_Pradesh", "Andhra_Pradesh", "Meghalaya"
]

In [18]:
def create_prompt():
    return f"""
You are an expert in Indian dialect identification.

Task:
Listen to the provided Hindi/English speech audio and predict the speaker's native state.

IMPORTANT RULES:
- Only choose ONE state from the allowed list
- Do NOT explain
- Do NOT output anything except the state name

Allowed States:
{', '.join(ALLOWED_STATES)}

Output format:
<State_Name>
"""

In [19]:
def predict_state(audio_path):
    prompt = create_prompt()

    response = model.generate_content(
        [
            prompt,
            {
                "mime_type": "audio/wav",
                "data": open(audio_path, "rb").read()
            }
        ]
    )

    return response.text.strip()

In [7]:
import pandas as pd

# Define the path to the speaker info file (adjust if necessary)
spkr_info_path = '../total_spkrinfo.list'
spkr_columns = ['Speaker_ID', 'Gender', 'Mother_Tongue', 'Height_cm',
                'Shoulder_size_cm', 'Waist_size_cm', 'Weight_kg', 'Age_y',
                'Native_State', 'Native_District']

# Load the speaker info into a DataFrame
spkr_info_df = pd.read_csv(spkr_info_path, sep=' ', names=spkr_columns)
spkr_info_df = spkr_info_df.drop(index=0)
print("Speaker Metadata (first 5 rows):")
print(spkr_info_df.head())

Speaker Metadata (first 5 rows):
  Speaker_ID  Gender Mother_Tongue Height_cm Shoulder_size_cm Waist_size_cm  \
1   Hin_0001  Female         Hindi       163               40          89.5   
2   Hin_0002  Female         Hindi     154.5             36.5            72   
3   Hin_0003    Male         Hindi     167.5             40.5            78   
4   Hin_0004    Male         Hindi       176               43          91.5   
5   Hin_0005  Female         Hindi       153             40.5            96   

  Weight_kg  Age_y    Native_State Native_District  
1      58.5  24.24       Rajasthan          Jaipur  
2      50.9  26.06  Madhya_Pradesh          Indore  
3      56.6  21.51         Haryana       Faridabad  
4      77.6  21.09    Chhattisgarh        Bilaspur  
5      80.2  27.39   Uttar_Pradesh    Kanpur_Nagar  


In [9]:
import os 
import glob

In [10]:
# Define the folder containing merged audio files
audio_folder = "../Merged_Audio"

# List all .wav files in the merged folder
audio_files = glob.glob(os.path.join(audio_folder, "*.wav"))

# Build a list to store audio file information
audio_data = []
for file_path in audio_files:
    file_name = os.path.basename(file_path)
    # Expected filename format: Hin_0067_Hin_m_0014.wav
    parts = file_name.split('_')
    if len(parts) >= 2:
        # Construct the full speaker ID from the first two tokens, e.g., "Hin_0067"
        speaker_id_full = f"{parts[0]}_{parts[1]}"

        audio_data.append({
            'Speaker_ID': speaker_id_full,

            'File_Path': file_path
        })
    else:
        print("Unexpected filename format:", file_name)

# Create a DataFrame from the audio data
audio_df = pd.DataFrame(audio_data)
print("Audio DataFrame (first 5 rows):")
print(audio_df.head())

Audio DataFrame (first 5 rows):
  Speaker_ID                                File_Path
0   Hin_0092  ../Merged_Audio/Hin_0092_Hin_f_0008.wav
1   Hin_0077  ../Merged_Audio/Hin_0077_Eng_f_7341.wav
2   Hin_0093  ../Merged_Audio/Hin_0093_Hin_f_0023.wav
3   Hin_0020  ../Merged_Audio/Hin_0020_Eng_m_5447.wav
4   Hin_0013  ../Merged_Audio/Hin_0013_Eng_m_5222.wav


In [11]:
audio_df['Speaker_ID'] = audio_df['Speaker_ID'].astype(str)
spkr_info_df['Speaker_ID'] = spkr_info_df['Speaker_ID'].astype(str)


merged_df = pd.merge(audio_df, spkr_info_df[['Speaker_ID', 'Native_State']],
                     on='Speaker_ID', how='left')

# Select only the desired columns: the audio file path and the corresponding Native_State.
final_df = merged_df[['File_Path', 'Speaker_ID', 'Native_State']]
print("Final Dataset (first 5 rows):")
print(final_df.head())

Final Dataset (first 5 rows):
                                 File_Path Speaker_ID   Native_State
0  ../Merged_Audio/Hin_0092_Hin_f_0008.wav   Hin_0092  Uttar_Pradesh
1  ../Merged_Audio/Hin_0077_Eng_f_7341.wav   Hin_0077    Maharashtra
2  ../Merged_Audio/Hin_0093_Hin_f_0023.wav   Hin_0093    Maharashtra
3  ../Merged_Audio/Hin_0020_Eng_m_5447.wav   Hin_0020  Uttar_Pradesh
4  ../Merged_Audio/Hin_0013_Eng_m_5222.wav   Hin_0013          Bihar


In [12]:
df= final_df
df.head()

,File_Path,Speaker_ID,Native_State
0,../Merged_Audio/Hin_0092_Hin_f_0008.wav,Hin_0092,Uttar_Pradesh
1,../Merged_Audio/Hin_0077_Eng_f_7341.wav,Hin_0077,Maharashtra
2,../Merged_Audio/Hin_0093_Hin_f_0023.wav,Hin_0093,Maharashtra
3,../Merged_Audio/Hin_0020_Eng_m_5447.wav,Hin_0020,Uttar_Pradesh
4,../Merged_Audio/Hin_0013_Eng_m_5222.wav,Hin_0013,Bihar


In [23]:
import pandas as pd

results = []

for i, row in df.iterrows():
    audio_path = row["File_Path"]
    true_label = row["Native_State"]

    try:
        pred = predict_state(audio_path)

        results.append({
            "true": true_label,
            "pred": pred
        })

        print(f"{i}: TRUE={true_label}, PRED={pred}")

    except Exception as e:
        print(f"Error at {i}: {e}")

Error at 0: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 38.382370478s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dime

KeyboardInterrupt: 